In [ ]:
# only needs to be run once per session
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py pyyaml

In [ ]:
# only needs to be run once per session
!apt-get install -y -q git-lfs
!git lfs install

In [ ]:
# !git clone https://github.com/noshou/APS360.git /kaggle/working/APS360  # first time only
!git -C /kaggle/working/APS360 pull                                        # run to get latest code

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/APS360")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # prevents fragmentation
os.environ["PYTHONPATH"] = "/kaggle/working/APS360"           # inherited by mp.spawn workers

# clear python cache
for mod in list(sys.modules.keys()):
    if "ScatterNet" in mod or "train" in mod:
        del sys.modules[mod]

from ScatterNet.config import RunConfig, DEFAULT_BUCKETS
from train import main

# bin 58 contains molecules up to 78,819 atoms; even with 2-GPU TP the shard
# (~39k atoms) exhausts T4 memory. Drop it — max molecule becomes 6,046 atoms.
SAFE_BUCKETS = [b for b in DEFAULT_BUCKETS if b[1] <= 6046]

cfg = RunConfig(

    # --- paths ---
    hdf5           = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5", # HDF5 dataset
    db             = "/kaggle/working/APS360/Preprocess/scatternet",     # SQLite encoding stem
    ckpt_best      = "/kaggle/working/scatternet_best.pt",               # saved on val improvement
    ckpt_resume    = "/kaggle/working/scatternet_resume.pt",             # saved every epoch
    metrics        = "/kaggle/working/scatternet_metrics.json",          # per-epoch loss + R2
    resume         = None,                                               # path to resume from, or None

    # --- model ---
    lambda_1       = 128,   # atom embedding dimension
    lambda_2       = 3,     # message passing rounds
    lambda_3       = 128,   # OutputHead hidden width
    lambda_4       = 4,     # MLP halving steps (2^lambda_4 <= lambda_3)
    lambda_5       = 128,   # Random Fourier Features
    msg_seed       = 42,    # RFF frequency matrix seed
    atm_chunk      = 32,    # atoms per M-chunk: controls RFF tensor peak (Nc, 32, Q, λ₅)
    mol_chunk      = 32,    # molecules per N-chunk: controls chem_env peak (32, Q, λ₅, λ₁) ≈ 104 MB
    eps_embd       = 1e-8,  # numerical floor in Embed
    eps_msgp       = 1e-3,  # numerical floor in MessagePass

    # --- loss ---
    lambda_6       = 0.1,   # form-factor penalty weight
    lambda_7       = 0.1,   # sigma inverse-L1 regularisation weight
    eps_sigma      = 1e-4,  # floor added to sigma before inverse-L1 penalty (prevents 1/sigma -> inf)

    # --- training ---
    lr             = 3e-4,  # Adam learning rate
    weight_decay   = 1e-5,  # Adam L2 weight decay
    grad_clip      = 1.0,   # max gradient norm
    epochs         = 50,    # epochs to train
    batcher_seed   = 0,     # train/val/test split seed
    atom_size_ceil = 78819, # max atoms per batch — MUST be large; small values create too many batches
    num_workers    = 3,     # DataLoader workers
    max_batches    = None,  # cap batches per epoch (None = no limit)
    use_amp        = True,  # mixed precision: halves activation memory via float16 forward

    # --- data ---
    buckets        = SAFE_BUCKETS,
)

main(cfg)